# Modelo melhorado: codificação das perdas e estimador, escolhidos por região

Este notebook fecha a investigação sobre melhoria do conjunto de dados e da modelagem. Ele implementa e mede as duas alterações que apresentaram ganho reprodutível, entre as vinte e duas estratégias avaliadas ao longo da pesquisa.

Nenhuma das duas cria dado novo. A primeira reorganiza informação que já estava na base, na coluna que registra o perfil de perda térmica. A segunda combina dois algoritmos já disponíveis.

## O que foi avaliado e descartado

Dezesseis estratégias de aumento sintético foram medidas e nenhuma apresentou ganho: duplicação, subamostragem, ruído no alvo e nas entradas, SMOTE em três variantes, SMOTER, mixup, réplica e peso junto ao limite, síntese por modos espaciais e permutação de alvos entre replicatas. Também foram descartados o treino sobre alvos imputados, que perde 0,397 grau, o uso de LightGBM com e sem restrição de monotonicidade, que perde mais de um grau, a previsão em dois estágios sobre a elevação, que perde 0,495 grau, e a busca de hiperparâmetros do Random Forest, que perde 0,103 grau por custo de seleção.

O método dos elementos finitos não integra o modelo. A investigação conduzida em notebook próprio mostrou que, em seis das sete regiões, o erro atribuível à estrutura espacial entre componentes situa-se abaixo do piso de ruído, de modo que não há o que um modelo físico dessa estrutura acrescente. A exceção é a região das bobinas, cuja aplicação depende de informações ainda não recebidas da parceira.

## As duas alterações adotadas

A primeira é a decomposição do rótulo de perda em família, ordem de acionamento e indicador de perfil modificado. Os rótulos originais são compostos, e a ordem de acionamento das bobinas, que é variável física, encontrava-se dissolvida dentro deles.

A segunda é a média simples entre Random Forest e Gradient Boosting. O teste confirmatório pré-registrado conduzido na região das bobinas apurou ganho de 0,813 grau, positivo nas dez sementes avaliadas, com o detalhe relevante de que o Gradient Boosting isolado é pior que o Random Forest: a média supera ambos, o que caracteriza complementaridade entre os erros e não redução de variância.

Ambas as escolhas são feitas por região, e a decisão é refeita dentro de cada dobra, de modo que o custo de escolher seja medido e não ignorado.

## 1. Configuração

A partição é lida do arquivo do projeto, e não sorteada aqui, de modo que todas as formulações sejam avaliadas exatamente sobre os mesmos ensaios. As três sementes existem para que nenhum ganho seja declarado a partir de um único sorteio da floresta.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S3/analise_dataset"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
from scipy import stats
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.preprocessing import StandardScaler

# A quarta sprint distribui apenas a codificacao de treze categorias; a de
# perfil de perdas permanece na pasta da terceira, e e de la que e lida.
B = "../../colabs_S4/"
B_PERFIL = "../"
part = json.load(open("../preparacao_testes/particao_treino_teste.json", encoding="utf-8"))
DOBRA = {int(k): v for k, v in part["dobra_por_run_idx"].items()}
PREF = ("blower_direction_", "housing_", "board_position_", "run_case_", "losses_",
        "perfil_perdas_", "deflector_hole_", "material_coil_support_", "cfg_",
          "reason_", "loss_profile_order_")
AREAS = ["oven_hmi", "ct_knobs", "main_vent_sys", "ipc_cooling",
         "mainback", "ipc_comp", "coil_temps"]
SEMENTES = [42, 1, 2]

## 2. As três codificações da coluna de perdas

A função abaixo monta, para cada região, o formato longo sob três codificações distintas da coluna de perdas, mantendo todo o resto igual. A codificação A é a de perfil de perdas da entrega anterior; a B são as treze categorias originais; e a C reorganiza os rótulos em família, ordem de chaveamento e marcador de modificação.

A codificação C é a razão de este caderno continuar existindo: ela não é percorrida pelas seções 21 e 23 do caderno de modelagem, de modo que a medição registrada aqui não existe em outro lugar.

In [ ]:
def prepara(area):
    dp = pd.read_csv(B_PERFIL + f"dataset_{area}_perfil.csv")
    d13 = pd.read_csv(B + f"dataset_{area}_13cat.csv")
    cols13 = [c for c in d13.columns if c.startswith("losses_")]
    rot = []
    for i in range(len(d13)):
        v = [c[len("losses_"):] for c in cols13 if d13.at[i, c] == 1]
        rot.append(v[0] if v else "")
    rot = pd.Series(rot)
    fam = rot.map(lambda r: "sem" if r == "" else ("nol" if "no losses" in r else
                  ("sim" if "simone" in r else ("mir" if "miro" in r else "out"))))
    ordm = rot.map(lambda r: "na" if (r == "" or "no losses" in r) else
                   ("f" if "front order" in r else ("b" if "back order" in r else "nd")))
    comuns = [c for c in dp.columns
              if c.startswith(PREF) and not c.startswith(("losses_", "perfil_perdas_"))]
    alvos = [c for c in dp.columns if not c.startswith(PREF) and c != "run_idx"]
    alvos = [c for c in alvos if dp[c].notna().sum() >= 5]
    cods = {
        "A": dp[[c for c in dp.columns if c.startswith("losses_")]].to_numpy(float),
        "B": d13[cols13].to_numpy(float),
        "C": np.hstack([pd.get_dummies(fam).to_numpy(float),
                        pd.get_dummies(ordm).to_numpy(float),
                        rot.str.contains("modified").astype(float).to_numpy().reshape(-1, 1)]),
    }
    base = dp[comuns].to_numpy(float)
    ens = np.arange(len(dp))
    saida = {}
    for nome, ext in cods.items():
        Xf = np.hstack([base, ext])
        Xs, ys, rs, cs = [], [], [], []
        for j, col in enumerate(alvos):
            ok = dp[col].notna().to_numpy()
            Xs.append(Xf[ok]); ys.append(dp[col].to_numpy()[ok])
            rs.append(ens[ok]); cs.append(np.full(ok.sum(), j))
        X = np.vstack(Xs); y = np.concatenate(ys)
        r = np.concatenate(rs); c = np.concatenate(cs)
        d = pd.Series(r).map(DOBRA).to_numpy()
        ok = ~pd.isna(d)
        saida[nome] = (X[ok], y[ok], r[ok], c[ok], d[ok])
    return saida, alvos

## 3. A grade e as quatro formulações

Para cada região, codificação, estimador e dobra, o erro de cada leitura é guardado com a sua chave. Sobre esse registro montam-se quatro formulações.

A formulação M0 é a entregue, com codificação de perfil e floresta aleatória. A M1 fixa a codificação por família e ordem em todas as regiões. A M2 escolhe a codificação por região, com o estimador fixo. A M3 escolhe codificação e estimador por região.

As escolhas de M2 e de M3 são refeitas dentro de cada dobra, com base apenas nas demais, sem o que o número mediria a própria escolha.

In [ ]:
resultado = []
for semente in SEMENTES:
    erro = {}          # (area, cfg) -> {chave: erro}
    for area in AREAS:
        cods, alvos = prepara(area)
        for cod, (X, y, r, c, d) in cods.items():
            for est in ("RF", "RF+GB"):
                erro[(area, f"{cod}|{est}")] = {}
            for k in sorted(set(d)):
                va = d == k
                tr = ~va
                if tr.sum() < 30 or not va.any():
                    continue
                pat = pd.Series(y[tr]).groupby(c[tr]).mean()
                esc = pd.Series(y[tr]).groupby(c[tr]).std().fillna(1.0).replace(0, 1.0)
                ger = float(y[tr].mean())

                def M(m):
                    return np.hstack([X[m],
                                      pd.Series(c[m]).map(pat).fillna(ger).to_numpy().reshape(-1, 1),
                                      pd.Series(c[m]).map(esc).fillna(1.0).to_numpy().reshape(-1, 1)])

                s = StandardScaler()
                Xt, Xv = s.fit_transform(M(tr)), s.transform(M(va))
                rf = RandomForestRegressor(n_estimators=300, random_state=semente,
                                           n_jobs=-1).fit(Xt, y[tr])
                gb = GradientBoostingRegressor(n_estimators=300,
                                               random_state=semente).fit(Xt, y[tr])
                prf, pgb = rf.predict(Xv), gb.predict(Xv)
                for i, pos in enumerate(np.where(va)[0]):
                    ch = (k, alvos[c[pos]], int(r[pos]))
                    erro[(area, f"{cod}|RF")][ch] = abs(prf[i] - y[pos])
                    erro[(area, f"{cod}|RF+GB")][ch] = abs((prf[i] + pgb[i]) / 2 - y[pos])
        print(f"  semente {semente} | {area} concluida", flush=True)

    # formulacoes
    form = {m: {} for m in ["M0", "M1", "M2", "M3"]}
    escolhas = {"M2": {}, "M3": {}}
    for area in AREAS:
        chaves = sorted(erro[(area, "A|RF")].keys())
        dobras = sorted({c[0] for c in chaves})
        for ch in chaves:
            form["M0"][(area,) + ch] = erro[(area, "A|RF")][ch]
            form["M1"][(area,) + ch] = erro[(area, "C|RF")][ch]
        for k in dobras:
            # M2: escolhe so a codificacao, estimador fixo em RF
            cand2 = {f"{c}|RF": np.mean([v for cc, v in erro[(area, f"{c}|RF")].items()
                                         if cc[0] != k]) for c in "ABC"}
            b2 = min(cand2, key=cand2.get)
            # M3: escolhe codificacao e estimador
            cand3 = {cfg: np.mean([v for cc, v in erro[(area, cfg)].items() if cc[0] != k])
                     for cfg in [f"{c}|{e}" for c in "ABC" for e in ("RF", "RF+GB")]}
            b3 = min(cand3, key=cand3.get)
            escolhas["M2"].setdefault(area, []).append(b2)
            escolhas["M3"].setdefault(area, []).append(b3)
            for ch in chaves:
                if ch[0] == k:
                    form["M2"][(area,) + ch] = erro[(area, b2)][ch]
                    form["M3"][(area,) + ch] = erro[(area, b3)][ch]

    comum = sorted(set.intersection(*[set(form[m]) for m in form]))
    resultado.append({"semente": semente,
                      **{m: float(np.mean([form[m][c] for c in comum])) for m in form},
                      "_e": {m: np.array([form[m][c] for c in comum]) for m in form},
                      "_chaves": comum, "_escolhas": escolhas})

## 4. Comparação entre as formulações

In [ ]:
tab = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")} for r in resultado])
print()
print("=" * 78)
print("MODELO FINAL, validacao cruzada por configuracao")
print("=" * 78)
print(tab.round(4).to_string(index=False))
print()
for m in ["M1", "M2", "M3"]:
    g = np.array([r["M0"] - r[m] for r in resultado])
    print(f"{m} sobre M0: ganho medio {g.mean():+.4f} C | por semente {np.round(g,4)} | "
          f"positivas {int((g>0).sum())}/{len(g)}")
print()
for a, b in [("M2", "M3"), ("M0", "M3")]:
    d = np.concatenate([r["_e"][a] - r["_e"][b] for r in resultado])
    bs = [np.random.default_rng(s).choice(d, len(d), replace=True).mean() for s in range(3000)]
    print(f"M3 contra {a}: ganho {d.mean():+.4f} C | "
          f"IC95 [{np.percentile(bs,2.5):+.4f}, {np.percentile(bs,97.5):+.4f}] | "
          f"wilcoxon p={stats.wilcoxon(np.concatenate([r['_e'][a] for r in resultado]), np.concatenate([r['_e'][b] for r in resultado])).pvalue:.2e}")

## 5. Escolha por região e erro por região

In [ ]:
print()
print("configuracao escolhida por regiao no M3 (semente 42):")
for area in AREAS:
    v = resultado[0]["_escolhas"]["M3"][area]
    cont = pd.Series(v).value_counts()
    print(f"  {area:<15} " + " | ".join(f"{a}: {b}/{len(v)}" for a, b in cont.items()))

print()
print("MAE por regiao:")
lin = []
for area in AREAS:
    d = {}
    for m in ["M0", "M3"]:
        vals = [e for r in resultado for c, e in zip(r["_chaves"], r["_e"][m]) if c[0] == area]
        d[m] = float(np.mean(vals))
    lin.append({"regiao": area, "M0 entregue": d["M0"], "M3 final": d["M3"],
                "ganho": d["M0"] - d["M3"]})
print(pd.DataFrame(lin).sort_values("ganho", ascending=False).round(3).to_string(index=False))

## Leitura dos resultados

A formulação M3 supera a entregue em todas as sementes avaliadas, e supera também a M2, que escolhe apenas a codificação. O ganho concentra-se na região das bobinas, que é simultaneamente a de maior erro do projeto e a de maior risco de decisão, por concentrar 33 das leituras que atingem o limite do componente.

Três regiões apresentam ganho negativo. Isso não constitui inconsistência: a configuração de cada dobra é decidida sem consultar essa dobra, de modo que em algumas delas a escolha erra. O agregado permanece positivo porque acerta na maioria, e é esse o número honesto.

A unanimidade da escolha em várias regiões é o que distingue seleção legítima de ajuste ao ruído. Em investigação anterior deste projeto, a seleção de algoritmo por componente apresentava o algoritmo mais escolhido em menos de um terço das escolhas, e o ganho aparente desapareceu sob medição aninhada.

## Limitações

O desempenho relatado provém de validação cruzada sobre o conjunto de treino. O conjunto reservado não foi consultado neste notebook. Cabe registrar que ele já foi utilizado uma vez, conforme consta da documentação, de modo que deixou de constituir árbitro plenamente independente.

O teste confirmatório da combinação de algoritmos utilizou as mesmas dobras da exploração que originou a hipótese. A hipótese e o critério de decisão foram declarados antes da execução, e o efeito mostrou-se grande e estável, mas confirmação independente exigiria dados não utilizados.

A decomposição do rótulo de perda decorre de leitura dos rótulos, e não de especificação fornecida pela parceira. A interpretação de que os termos front order e back order designam a sequência de acionamento das bobinas é consistente com os dados, mas carece de confirmação. Confirmada também a hipótese de que o rótulo simone corresponde ao perfil de potência de 240, 145, 180 e 180 watts, a família do perfil poderia ser substituída por valores numéricos de potência em 32 dos 56 ensaios, o que representaria ganho de outra ordem de grandeza.

O piso de ruído ponderado da base situa-se em 2,79 graus, contra o erro atual do modelo. A folga remanescente não decorre da formulação, e sim da ausência, entre as variáveis disponíveis, da grandeza que determina o nível de temperatura de cada ensaio.